# DDNN dataset preparation — 2015-20 and 2020-26

Interactive front-end for the ETL in `data_pipeline/` (CLI equivalent: `python build_datasets.py`).

It builds two hourly datasets with the exact schema of `Datasets/DE.csv` (Marcjasz et al., 2022):

| Output | Window | Power columns | Daily columns |
|---|---|---|---|
| `Datasets/DE_dataset_2015-2020.csv` | 2015-01-01 → 2020-12-31 | copied from `DE.csv` (DE-AT-LU split on 2018-10-01) | raw files in `2015-20/` (DE.csv fallback before the first raw quote) |
| `Datasets/DE_dataset_2020-2026.csv` | 2020-11-19 → 2026-11-19 | SMARD files in `2020-26/` | raw files in `2020-26/` |

**Rules applied**
- One row = delivery hour starting at the timestamp; **no lag is baked in** (the models build D-1/D-2/D-7 themselves by row position).
- Exactly 24 rows/day: March missing 02:00 → interpolated; October duplicate 02:00 → averaged.
- Daily series: continuous calendar, forward-fill weekends/holidays, USD→EUR (coal, Brent) **after** ffill, then broadcast to 24 h.
- SMARD gaps: ≤3 h linear interpolation; whole missing days → same hour one week earlier/later.
- `Load_DA_Forecast` = *grid load* (not residual load); `Renewables_DA_Forecast` = *Photovoltaics and wind*.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import data_pipeline as dp
from data_pipeline import config, parsing

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
DATA_DIR = config.DATA_DIR
DATA_DIR

## 1. Configuration
Edit `data_pipeline/config.py` to change windows, file patterns or fallback behaviour.

In [ ]:
pd.DataFrame([vars(w) for w in config.WINDOWS.values()]).drop(columns="notes")

## 2. Peek at the raw parsing
Each reader auto-detects the delimiter, number format (`1,234.56` vs `1.234,56`), date format and missing tokens (`-`, `.`).

In [ ]:
folder = DATA_DIR / "2020-26"
price_raw = parsing.read_smard(parsing.find_file(folder, config.SMARD_SOURCES["Price"].pattern), "Germany/Luxembourg")
print("SMARD price rows:", len(price_raw), "| duplicated timestamps (DST fall-back):", price_raw.index.duplicated().sum())
price_raw.loc["2022-03-27 00:00":"2022-03-27 04:00"]   # spring-forward: no 02:00 row in the raw file

In [ ]:
# number parser: English and German notation, '-' / '.' as missing
pd.DataFrame({"english": parsing.parse_number_series(pd.Series(["2,712.99", "-", "81.25"])),
              "german":  parsing.parse_number_series(pd.Series(["2.712,99", ".", "81,25"]))})

In [ ]:
coal = parsing.read_investing(parsing.find_file(folder, config.DAILY_SOURCES["API2_Coal"].pattern))
fx = dp.load_fx(DATA_DIR)
pd.concat({"coal_USD": coal, "EURUSD": fx}, axis=1).loc["2020-11-19":"2020-11-25"]

## 3. Build the datasets

In [ ]:
reference = dp.load_reference(DATA_DIR)
results = {name: dp.build_window(cfg, DATA_DIR, reference=reference, fx=fx)
           for name, cfg in config.WINDOWS.items()}

### Provenance & filling report — 2015-20
`days_from_reference` = days before the first raw quote (in `2015-20/`, TTF starts 2017-10-23 and coal 2020-01-02) that were taken from `DE.csv`.

In [ ]:
dp.report_frame(results["2015-20"])

### Provenance & filling report — 2020-26

In [ ]:
dp.report_frame(results["2020-26"])

### Lag check of the daily series vs `DE.csv`
`best_lag_days = 0` confirms that DE.csv stores day *T*'s close on day *T* (the model applies the D-2 shift itself).

In [ ]:
pd.concat({n: dp.alignment_table(reference, r.daily_eur, r.daily_has_quote) for n, r in results.items()})

### DST days in the 2020-26 output (24 rows each)

In [ ]:
out = results["2020-26"].data
print("rows per day:", out.groupby(out.index.normalize()).size().unique())
out.loc["2022-03-27 00:00":"2022-03-27 04:00", ["Price", "Load_DA_Forecast"]]   # 02:00 interpolated

In [ ]:
out.loc["2022-10-30 00:00":"2022-10-30 04:00", ["Price", "Load_DA_Forecast"]]   # 02:00 = mean of the two raw 02:00

## 4. Write the CSV files

In [ ]:
for r in results.values():
    print(dp.write_dataset(r, DATA_DIR))

## 5. Validation
Integrity, overlap 2020-11-19 → 2020-12-31 against `DE.csv`, summary statistics.

Power columns of the 2020-26 file differ slightly from `DE.csv` on the overlap (SMARD vs paper data): accepted trade-off. A `CHECK UNITS` flag would indicate a scaling / currency / lag problem.

In [ ]:
checks = dp.validate_outputs(DATA_DIR, verbose=False)
pd.DataFrame(checks["integrity"]).T

In [ ]:
checks["overlap"]

In [ ]:
checks["summary"].round(2)